# Bloque 4 — Agentes de IA
## Notebook 00: De la llamada al modelo al equipo de agentes

---

### ¿Qué hemos hecho hasta aquí?

En los módulos anteriores hemos usado LLMs como **clasificadores**: le pasamos un texto, nos devuelve una etiqueta. El modelo no recuerda nada, no toma decisiones, no puede pedir más información. Es como un formulario que rellena solo.

```
[Texto] → LLM → [Categoría]
```

### ¿Qué es un agente?

Un **agente** es un LLM al que le añadimos tres capacidades:

| Capacidad | Qué significa |
|---|---|
| **Rol** | El modelo sabe quién es y cuál es su objetivo |
| **Memoria** | Puede recordar lo que ha descubierto en la conversación |
| **Herramientas** | Puede llamar funciones externas (buscar en una BD, ejecutar código) |

El loop de un agente es:

```
┌─────────────────────────────────────────────┐
│                   AGENTE                    │
│                                             │
│  1. PERCIBE   → Lee la tarea y el contexto  │
│  2. RAZONA    → Piensa qué hacer (CoT)      │
│  3. ACTÚA     → Llama herramienta / escribe │
│  4. OBSERVA   → Ve el resultado de la acción│
│  5. Repite hasta completar la tarea         │
└─────────────────────────────────────────────┘
```

### ¿Qué es una crew (equipo)?

Una **crew** es un grupo de agentes especializados que colaboran. Cada uno tiene un rol concreto y sus outputs alimentan al siguiente:

```
[Investigador] ──→ [Analista] ──→ [Redactor] ──→ Informe final
   qwen3.5        qwen2.5:14b    deepseek-r1
```

Esto es exactamente lo que construiremos en el notebook 01.

## Los tres primitivos de CrewAI

CrewAI organiza todo en tres conceptos:

### `Agent` — quién es
```python
Agent(
    role="Analista de amenazas",          # su título profesional
    goal="Identificar actores clave",      # su objetivo general
    backstory="Llevas 10 años en CTI...", # contexto que moldea su comportamiento
    llm=...,                               # el modelo que lo alimenta
)
```

### `Task` — qué tiene que hacer
```python
Task(
    description="Analiza estos perfiles y lista los 5 actores...", # instrucciones
    expected_output="Una lista numerada con nombre y justificación", # formato esperado
    agent=analista,   # quién la ejecuta
)
```

### `Crew` — el equipo completo
```python
Crew(
    agents=[investigador, analista, redactor],
    tasks=[tarea1, tarea2, tarea3],
    process=Process.sequential,  # en orden, el output de cada tarea pasa a la siguiente
)
resultado = crew.kickoff()        # ¡a trabajar!
```

## Nuestra distribución de modelos

Una de las ventajas de CrewAI es que **cada agente puede usar un modelo distinto**. No necesitamos el mismo LLM para todo: podemos asignar el modelo más adecuado a cada tipo de tarea y equilibrar velocidad vs. razonamiento.

En este bloque usamos tres modelos Ollama locales:

| Agente | Modelo | Por qué este modelo |
|---|---|---|
| `investigador` | `qwen3.5:latest` | Tarea de extracción y búsqueda — modelo ágil, responde rápido |
| `analista` | `qwen2.5:14b` | Análisis de patrones y datos estructurados — sólido en razonamiento tabular |
| `redactor` | `deepseek-r1:14b` | Síntesis y redacción de informes — R1 razona internamente (CoT) antes de escribir |

> **Nota sobre DeepSeek R1**: el modelo emite un bloque `<think>...</think>` con su cadena de razonamiento antes de la respuesta final. CrewAI lo filtra automáticamente del output que pasa al siguiente agente, pero lo verás en los logs si tienes `verbose=True`.

En este notebook 00 usamos `qwen3.5` para el agente de demostración — es el más ligero y va más rápido mientras aprendemos los conceptos. Los tres entran en juego a partir del notebook 01.

In [ ]:
# ─── VERIFICACIÓN DEL ENTORNO ────────────────────────────────────────────────
# Antes de empezar comprobamos que las librerías necesarias están instaladas.
# Si alguna falla aquí, revisar que has hecho `uv sync` en la raíz del repo.

import importlib

for libreria in ['crewai', 'ollama']:
    try:
        mod = importlib.import_module(libreria)
        version = getattr(mod, '__version__', 'desconocida')
        print(f'✓ {libreria} {version}')
    except ImportError:
        print(f'✗ {libreria} NO encontrado — ejecuta: uv sync')

In [ ]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import concurrent.futures

# CrewAI necesita una variable de entorno OPENAI_API_KEY para inicializarse,
# incluso cuando no usamos OpenAI. Le pasamos un valor falso — nunca se enviará
# a ningún servidor externo porque configuramos Ollama como backend local.
os.environ['OPENAI_API_KEY'] = 'NA'

# Los tres primitivos de CrewAI: el agente, la tarea y el equipo.
from crewai import Agent, Task, Crew, Process

# LLM es la clase que envuelve cualquier modelo para usarlo en CrewAI.
# El prefijo "ollama/" activa el adaptador LiteLLM para Ollama local.
from crewai import LLM

import ollama


def kickoff(crew, timeout=300):
    """Ejecuta crew.kickoff() en un hilo separado.

    CrewAI 1.x detecta el event loop de Jupyter y lanza RuntimeError si se llama
    kickoff() directamente desde un notebook. En un hilo nuevo el hilo no hereda
    el event loop activo y CrewAI puede crear el suyo propio sin conflicto.
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


print('Imports correctos.')

In [ ]:
# ─── CONFIGURACIÓN DE LOS TRES LLMs ──────────────────────────────────────────
# Definimos un objeto LLM por cada modelo del bloque.
# Todos apuntan al mismo Ollama local (localhost:11434) pero usan modelos distintos.
# En los notebooks 01 y 02 asignaremos cada llm_* al agente que le corresponde.

OLLAMA_URL = 'http://localhost:11434'

# Agente investigador — extracción rápida de datos
llm_investigador = LLM(
    model='ollama/qwen3.5:latest',
    base_url=OLLAMA_URL,
)

# Agente analista — razonamiento sobre datos estructurados
llm_analista = LLM(
    model='ollama/qwen2.5:14b',
    base_url=OLLAMA_URL,
)

# Agente redactor — síntesis y redacción con chain-of-thought interno (R1)
llm_redactor = LLM(
    model='ollama/deepseek-r1:14b',
    base_url=OLLAMA_URL,
)

print('LLMs configurados:')
print(f'  investigador → {llm_investigador.model}')
print(f'  analista     → {llm_analista.model}')
print(f'  redactor     → {llm_redactor.model}')

In [ ]:
# ─── TEST DE CONECTIVIDAD ─────────────────────────────────────────────────────
# Comprobamos que los tres modelos responden antes de continuar.
# Si alguno falla, verificar que Ollama está corriendo: `ollama serve`
# y que el modelo está descargado: `ollama list`

MODELOS = {
    'investigador': 'qwen3.5:latest',
    'analista':     'qwen2.5:14b',
    'redactor':     'deepseek-r1:14b',
}

for rol, modelo in MODELOS.items():
    try:
        resp = ollama.chat(
            model=modelo,
            messages=[{'role': 'user', 'content': 'Responde solo: OK'}],
        )
        contenido = resp['message']['content'].strip()[:60]
        print(f'✓ {rol:15s} ({modelo}) → {contenido}')
    except Exception as e:
        print(f'✗ {rol:15s} ({modelo}) → ERROR: {e}')

## Nuestro primer agente

Vamos a crear un agente muy sencillo: un analista de ciberseguridad al que le hacemos una pregunta sobre el grupo Conti. Sin datos externos, sin herramientas — solo el conocimiento que ya tiene el modelo.

Usamos `qwen3.5` (el modelo del investigador) porque es el más rápido para este ejemplo introductorio.

In [ ]:
# ─── DEFINIR EL AGENTE ────────────────────────────────────────────────────────
# El `role` es como el título en una tarjeta de visita: marca la especialidad.
# El `goal` es el objetivo que guía todas las decisiones del agente.
# El `backstory` da contexto que moldea el tono y profundidad de las respuestas.
# Cuanto más específico sea el backstory, más coherente será el comportamiento.

analista_conti = Agent(
    role='Analista de Threat Intelligence especializado en ransomware',
    goal='Proporcionar análisis precisos y útiles sobre grupos de ransomware '
         'basándose en los datos disponibles y el conocimiento del dominio.',
    backstory=(
        'Eres un analista senior de CTI (Cyber Threat Intelligence) con 8 años '
        'de experiencia rastreando grupos de ransomware. Has estudiado en '
        'profundidad el grupo Conti, sus métodos de operación y estructura interna. '
        'Respondes siempre en español, con precisión técnica y sin exagerar.'
    ),
    llm=llm_investigador,  # usamos qwen3.5 para este agente de demo
    verbose=True,           # con verbose=True veremos el razonamiento interno
)

In [ ]:
# ─── DEFINIR LA TAREA Y LANZAR EL CREW ────────────────────────────────────────
# La `description` es lo que el agente tiene que hacer, con todo el contexto necesario.
# El `expected_output` le dice al agente exactamente qué formato debe tener su respuesta.
# Sin expected_output, el agente puede dar respuestas demasiado largas o mal formateadas.

tarea_intro = Task(
    description=(
        'Explica brevemente qué fue el grupo de ransomware Conti: '
        'cuándo operó, cómo estaba organizado y por qué fue significativo. '
        'Limita tu respuesta a lo que es factualmente verificable.'
    ),
    expected_output=(
        'Un párrafo de entre 100 y 150 palabras con: '
        '(1) período de actividad, (2) estructura organizativa, (3) impacto relevante.'
    ),
    agent=analista_conti,
)

# El Crew une agentes y tareas. Process.sequential ejecuta las tareas en orden.
# Con un solo agente y una sola tarea es equivalente a una llamada directa al LLM,
# pero con la estructura que escalaremos a múltiples agentes en el notebook 01.
crew_intro = Crew(
    agents=[analista_conti],
    tasks=[tarea_intro],
    process=Process.sequential,
    verbose=True,
)

resultado = kickoff(crew_intro)

print('\n' + '=' * 60)
print('RESULTADO FINAL:')
print('=' * 60)
print(resultado.raw)

## Anatomía del output

Con `verbose=True` ves el loop interno del agente:

```
> Entering new AgentExecutor chain...
Thought: Tengo que explicar el grupo Conti...   ← el modelo razona
Final Answer: El grupo Conti fue...              ← la respuesta final
```

Esto es el **Chain of Thought (CoT)**: el modelo "piensa en voz alta" antes de responder. `deepseek-r1` lleva esto más lejos con un bloque `<think>` explícito — lo veremos en el notebook 01.

Con `verbose=False` solo ves el resultado final — útil en producción, pero menos instructivo para aprender.

In [ ]:
# ─── EXPERIMENTO: verbose=False ───────────────────────────────────────────────
# Misma pregunta, mismo agente, pero sin mostrar el razonamiento intermedio.
# Compara los tiempos y el output — el resultado es idéntico, pero más silencioso.

analista_silencioso = Agent(
    role=analista_conti.role,
    goal=analista_conti.goal,
    backstory=analista_conti.backstory,
    llm=llm_investigador,
    verbose=False,  # sin log de razonamiento
)

crew_silencioso = Crew(
    agents=[analista_silencioso],
    tasks=[
        Task(
            description=tarea_intro.description,
            expected_output=tarea_intro.expected_output,
            agent=analista_silencioso,
        )
    ],
    process=Process.sequential,
    verbose=False,
)

resultado2 = kickoff(crew_silencioso)
print('Resultado (sin verbose):')
print(resultado2.raw)

## Resumen

| Concepto | Para qué sirve |
|---|---|
| `Agent` | Define quién es el modelo: rol, objetivo, personalidad |
| `Task` | Define qué tiene que hacer y qué formato debe tener el output |
| `Crew` | Une agentes y tareas; controla el orden de ejecución |
| `Process.sequential` | Ejecuta tareas en orden; el output de cada una pasa a la siguiente |
| `verbose=True` | Muestra el razonamiento interno — útil para depurar y aprender |

### Distribución de modelos en este bloque

| Agente | Modelo | Por qué este modelo |
|---|---|---|
| `investigador` | `qwen3.5:latest` | Extracción rápida de datos — modelo ágil |
| `analista` | `qwen2.5:14b` | Análisis de patrones y datos estructurados |
| `redactor` | `deepseek-r1:14b` | Síntesis con razonamiento CoT explícito (R1) |

**Siguiente**: en `01_crew_investigacion.ipynb` ponemos a trabajar los tres agentes juntos sobre datos reales de ContiLeaks.